# 02. Chat Model 호출

LangChain에서 OpenAI ChatModel을 불러오고 기본 호출 방법을 실습합니다.

## 1. 환경 설정

In [8]:
import os
import sys
from dotenv import load_dotenv
sys.path.append("..")  # 상위 폴더의 common_config.py를 import하기 위해 추가
from common_config import get_llm

load_dotenv(override=True, dotenv_path="../.env")

True

## 2. ChatOpenAI 초기화

- `model`: 사용할 모델 이름
- `temperature`: 0에 가까울수록 일관된 답변, 1에 가까울수록 창의적인 답변

In [9]:
# MonoRouter 경유 (gpt-4o-mini는 MonoRouter에서 지원하지 않음)
llm = get_llm("gpt-5.4-mini", temperature=0)
print(llm)

metadata={'lc_versions': {'langchain-core': '1.6.6', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}} profile={'name': 'GPT-5.4 mini', 'release_date': '2026-03-17', 'last_updated': '2026-03-17', 'open_weights': False, 'max_input_tokens': 400000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh']} client=<openai.resources.chat.completions.completions.Completions object at 0x744ca3392180> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x744ca33917c0> root_cl

## 3. 기본 호출 - invoke()

문자열을 직접 전달하면 HumanMessage로 자동 변환됩니다.

In [10]:
response = llm.invoke("LangChain이 무엇인지 한 문장으로 설명해줘")
print(response)

content='LangChain은 **대규모 언어 모델(LLM)을 이용한 앱을 쉽게 만들 수 있도록 도와주는 개발 프레임워크**입니다.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 18, 'total_tokens': 54, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini', 'system_fingerprint': None, 'id': 'chatcmpl-EU3HUKsE6Jy9IaVGvWUmjYouwGhYI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a0f5da-ea59-7393-b55e-ed135efeafda-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 18, 'output_tokens': 36, 'total_tokens': 54, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [11]:
# content만 출력
print(response.content)

LangChain은 **대규모 언어 모델(LLM)을 이용한 앱을 쉽게 만들 수 있도록 도와주는 개발 프레임워크**입니다.


## 4. 메시지 객체로 호출

역할(role)을 구분하여 메시지를 전달할 수 있습니다.
- `SystemMessage`: AI의 역할/페르소나 설정
- `HumanMessage`: 사용자 입력

In [12]:
from langchain_core.messages import SystemMessage, HumanMessage

messages = [
    SystemMessage(content="당신은 친절한 파이썬 튜터입니다."),
    HumanMessage(content="리스트 컴프리헨션을 초보자에게 설명해줘")
]

response = llm.invoke(messages)
print(response.content)

물론이죠!  
**리스트 컴프리헨션(list comprehension)**은 파이썬에서 **리스트를 간단하게 만드는 문법**입니다.  
초보자에게는 “`for`문을 한 줄로 줄여 쓴 것”이라고 생각하면 이해하기 쉬워요.

---

## 1) 기본 개념

예를 들어, 0부터 4까지 숫자를 담은 리스트를 만들고 싶다면:

```python
numbers = []
for i in range(5):
    numbers.append(i)
```

이걸 리스트 컴프리헨션으로 쓰면:

```python
numbers = [i for i in range(5)]
```

결과는 똑같이:

```python
[0, 1, 2, 3, 4]
```

---

## 2) 구조

리스트 컴프리헨션의 기본 형태는 이렇습니다:

```python
[표현식 for 변수 in 반복가능한것]
```

예:

```python
[i * 2 for i in range(5)]
```

의미:
- `range(5)`에서 숫자를 하나씩 꺼내서
- `i * 2`를 계산하고
- 그 결과를 리스트로 만듭니다

결과:

```python
[0, 2, 4, 6, 8]
```

---

## 3) 조건도 넣을 수 있어요

특정 조건에 맞는 것만 리스트에 넣고 싶을 때도 자주 씁니다.

예를 들어, 짝수만 남기기:

```python
[i for i in range(10) if i % 2 == 0]
```

결과:

```python
[0, 2, 4, 6, 8]
```

의미:
- `range(10)`의 숫자 중에서
- `i % 2 == 0`인 것만 골라서
- 리스트에 넣는다

---

## 4) 일반 for문과 비교

예를 들어, 숫자의 제곱을 담고 싶다면:

### 일반 for문
```python
squares = []
for i in range(5):
    squares.append(i ** 2)
```

### 리스트 컴프리헨션
```python
squares = [i ** 2 for i in range(

- 마크다운으로 출력해서 보기

In [13]:
from IPython.display import Markdown, display
display(Markdown(response.content))

물론이죠!  
**리스트 컴프리헨션(list comprehension)**은 파이썬에서 **리스트를 간단하게 만드는 문법**입니다.  
초보자에게는 “`for`문을 한 줄로 줄여 쓴 것”이라고 생각하면 이해하기 쉬워요.

---

## 1) 기본 개념

예를 들어, 0부터 4까지 숫자를 담은 리스트를 만들고 싶다면:

```python
numbers = []
for i in range(5):
    numbers.append(i)
```

이걸 리스트 컴프리헨션으로 쓰면:

```python
numbers = [i for i in range(5)]
```

결과는 똑같이:

```python
[0, 1, 2, 3, 4]
```

---

## 2) 구조

리스트 컴프리헨션의 기본 형태는 이렇습니다:

```python
[표현식 for 변수 in 반복가능한것]
```

예:

```python
[i * 2 for i in range(5)]
```

의미:
- `range(5)`에서 숫자를 하나씩 꺼내서
- `i * 2`를 계산하고
- 그 결과를 리스트로 만듭니다

결과:

```python
[0, 2, 4, 6, 8]
```

---

## 3) 조건도 넣을 수 있어요

특정 조건에 맞는 것만 리스트에 넣고 싶을 때도 자주 씁니다.

예를 들어, 짝수만 남기기:

```python
[i for i in range(10) if i % 2 == 0]
```

결과:

```python
[0, 2, 4, 6, 8]
```

의미:
- `range(10)`의 숫자 중에서
- `i % 2 == 0`인 것만 골라서
- 리스트에 넣는다

---

## 4) 일반 for문과 비교

예를 들어, 숫자의 제곱을 담고 싶다면:

### 일반 for문
```python
squares = []
for i in range(5):
    squares.append(i ** 2)
```

### 리스트 컴프리헨션
```python
squares = [i ** 2 for i in range(5)]
```

리스트 컴프리헨션이 더 짧고 읽기 쉬운 경우가 많습니다.

---

## 5) 언제 쓰면 좋을까?

리스트 컴프리헨션은 다음처럼 **간단한 작업**에 좋습니다:

- 숫자 변환
- 조건에 맞는 요소만 선택
- 문자열 처리
- 새로운 리스트 빠르게 만들기

예:

```python
words = ["apple", "banana", "cat"]
lengths = [len(word) for word in words]
```

결과:

```python
[5, 6, 3]
```

---

## 6) 초보자가 주의할 점

리스트 컴프리헨션은 편리하지만,  
**너무 복잡하게 쓰면 오히려 읽기 어려워질 수 있어요.**

예를 들어 이런 코드는 초보자에게 어렵습니다:

```python
[(x, y) for x in range(3) for y in range(3) if x != y]
```

이럴 땐 일반 `for`문이 더 이해하기 쉬울 수 있어요.

---

## 7) 한 줄 요약

리스트 컴프리헨션은:

> **“for문과 if문을 이용해 리스트를 간단하게 만드는 파이썬 문법”**

입니다.

---

원하시면 다음으로  
1. **리스트 컴프리헨션 예제 10개**  
2. **for문과 비교한 그림 설명**  
3. **딕셔너리/세트 컴프리헨션**  
중 하나로 이어서 설명해드릴게요.

## 5. 스트리밍 호출 - stream()

- 토큰이 생성되는 즉시 출력함.
- llm 객체에 질문을 보내고, 응답을 한 번에 받지 않고 조각(chunk) 단위로 순차적으로 받음.
- end=""를 쓰면 줄바꿈 없이 이어서 출력함.
- flush=True는 출력 버퍼를 즉시 비우라는 의미임.

In [14]:
for chunk in llm.stream("최신 langchain 버전을 알여주고, 특장점을 알여줘 "):
    print(chunk.content, end="", flush=True)

현재 제 지식 기준으로는 **LangChain의 “최신 정확한 버전 번호”를 실시간으로 확인할 수는 없습니다.**  
버전은 자주 업데이트되므로, 가장 정확한 값은 아래처럼 확인하는 것이 좋습니다.

### 최신 버전 확인 방법
- **PyPI**
  - `pip index versions langchain`
  - 또는 `pip install langchain --upgrade`
- **공식 GitHub / 릴리스 노트**
  - LangChain GitHub Releases 확인

---

## LangChain의 주요 특장점

### 1) LLM 앱 개발을 빠르게 해줌
LangChain은 LLM을 이용한 앱을 만들 때 필요한 요소들을 표준화해 줍니다.
- 프롬프트 관리
- 체인 구성
- 메모리
- 검색(RAG)
- 에이전트
- 도구 연동

즉, **“LLM 호출”만 하는 것이 아니라, 실제 서비스 형태로 묶는 작업**을 쉽게 합니다.

---

### 2) RAG 구현이 편리함
문서 검색 기반 생성(Retrieval-Augmented Generation)을 만들기 좋습니다.
- 벡터 저장소 연동
- 문서 로딩/분할
- 검색기(retriever) 구성
- 검색 결과를 프롬프트에 결합

이 덕분에 사내 문서, PDF, 위키, DB 등을 활용하는 챗봇을 빠르게 만들 수 있습니다.

---

### 3) 다양한 모델/도구와 연결 가능
OpenAI뿐 아니라 여러 LLM 공급자와 연동할 수 있고, 외부 도구와도 연결이 쉽습니다.
- OpenAI, Anthropic, Google 등
- 검색 엔진
- 데이터베이스
- API
- 함수 호출(function calling) 스타일 도구

---

### 4) 에이전트 구현이 쉬움
LLM이 상황에 따라 어떤 도구를 사용할지 결정하는 **에이전트 기반 로직**을 구현할 수 있습니다.
예:
- “웹 검색해줘”
- “DB에서 조회해줘”
- “계산해줘”
- “문서 요약해줘”

---

### 5) 체계적인 컴포넌트 구조
LangC

## 6. 응답 메타데이터 확인

토큰 사용량 등 메타정보를 확인할 수 있습니다.

In [15]:
response = llm.invoke("안녕")
print("content:", response.content)
print("usage:", response.usage_metadata)

content: 안녕하세요! 무엇을 도와드릴까요?
usage: {'input_tokens': 8, 'output_tokens': 14, 'total_tokens': 22, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
